# AIC PREPROCESS LAB #
## Basic information ##
- Model name: CLIP-ViT-H14 (DFN5B finetuned by Apple)
- Databases: AIC24, AIC25
- Store at: AWS S3 Service
- Vector database: Milvus

**ESSENTIAL INSTALLATION**

**SETUP**

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image
from open_clip import create_model_from_pretrained

print('Getting model...')
model, preprocess = create_model_from_pretrained('hf-hub:apple/DFN5B-CLIP-ViT-H-14-384')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
model.eval()
model.to(device)

c:\Users\Bui Thien Nghia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Getting model...
Using device: cuda


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwise_affine=Tru

**UTILS FUNCTION DEFINING**

In [ ]:
# def load_dataset_from_s3(bucket_name: str, obj_key: str):
#   """
#   Loads dataset from a S3 Bucket.

#   Arguments:
#     - bucket_name (str): Name of the bucket
#     - obj_key (str): URI of the object. Object must be in pickle format

#   Returns:
#     - dataset (Any): Dataset loaded from PKL file
#   """
#   print(f'Loading dataset from bucket: {bucket_name}')

#   response = s3.get_object(Bucket=bucket_name, Key=obj_key)
#   body = BytesIO(response['Body'].read())
#   dataset = torch.load(body)

#   print(f'Done loading dataset! Total samples count: {len(dataset)}\n')
#   return dataset


def batch_dataset(dataset, batch_size=128):
  print(f'Batching datasett, batch size: {batch_size}')
  batches = []
  for i in range(0, len(dataset), batch_size):
    batch = dataset[i:i+batch_size] if i < len(dataset) else dataset[i:len(dataset)]
    batches.append(batch)

  print(f'Done patching! Number of batches count: {len(batches)}\n')
  return batches


def preprocess_batch(batch: list[str]):
    images = []
    for link in batch:
        try:
          img = Image.open(f'D:\\AIC2025\\{link.split('/', 4)[-1].replace('/','\\')}')
        except FileNotFoundError:
          print(f'No such file: D:\\AIC2025\\{link.split('/', 4)[-1].replace('/','\\')}')
          continue

        img_preprocessed = preprocess(img).unsqueeze(0)
        images.append(img_preprocessed)
    
    return torch.cat(images)


def insert_vector_batch(dataset: dict, key_batch: list, vector_batch: list):
  for key, vector in zip(key_batch, vector_batch):
      dataset[key]['vector'] = vector


def tensor_to_vector_list(tensor: torch.Tensor):
    return [row.tolist() for row in tensor]


def compute_batch_features(model, key_batch: list, **options):
  """
  Compute features for all image keys in the batch using given model.

  Arguments:
    - model (open_clip.model.CLIP): CLIP model from OpenCLIP used for feature 
    extraction
    - key_batch (list): list of image keys/path to encode
    **options:
      - preprocessed_batch (list[torch.Tensor]): By default, the function will
      automatically preprocess images for you based on your key_batch. If you
      have a list of preprocessed image, the function will use it instead.

      - auto_insert_into (dict): By default, the function will return the image
      feature batch. If you want to automatically insert vectors into dataset,
      pass your dataset as a dictionary with keys are items in
      key_batch and values are dictionaries with 'vector' key. The code will
      return None then.

  Returns:
    - image_feature_batch (torch.Tensor): Image feature batch tensor with
    dimension (batch, vector_length), or
    - None: if auto_insert_into is True
  """
  with torch.no_grad(), torch.amp.autocast('cuda'):
    if 'preprocessed_batch' in options:
      batch_preprocessed = options['preprocessed_batch']
    else:
      batch_preprocessed = preprocess_batch(key_batch)

    image_feature_batch = model.encode_image(batch_preprocessed.to(device))
    image_feature_batch = F.normalize(image_feature_batch, dim=-1)

  # if device == 'cuda':
  #   torch.cuda.empty_cache()

  if 'auto_insert_into' in options:
    insert_vector_batch(options['auto_insert_into'], key_batch, tensor_to_vector_list(image_feature_batch.cpu()))
    return None
  else:
    return image_feature_batch

**SETTING UP DATASET**

In [1]:
dataset = torch.load('aic_2025.pt')

key_batches = batch_dataset(list(dataset.keys()), 32)

NameError: name 'torch' is not defined

**COMPUTING & ADDING FEATURE VECTORS TO DATASET**

In [13]:
print(f'Using {device} to compute')
with tqdm(key_batches, desc='Computing batches') as t:
  for batch in t:
    compute_batch_features(model, batch, auto_insert_into=dataset)

    elapsed = t.format_dict['elapsed']
    elapsed_str = t.format_interval(elapsed)

Using cuda to compute


Computing batches: 100%|██████████| 6406/6406 [4:40:29<00:00,  2.63s/it]  


In [25]:
torch.cuda.empty_cache()

**DATA SAVING**

In [14]:
torch.save(dataset, 'aic_2025-wfeat-b2.pt')

**DEBUGGING**

In [27]:
key, value = list(dataset.items())[255]
print(f'Key: {key}\nValue: {value}')

Key: kfs/Keyframes_L21/L21_V001/256.jpg
Value: {'img_key': 'kfs/Keyframes_L21/L21_V001/256.jpg', 'vector': [0.0039010639302432537, 0.022820454090833664, 0.0018291055457666516, 0.005007393192499876, -0.012080962769687176, -0.013013825751841068, 0.02067718096077442, 0.01099390722811222, 0.03342888131737709, -0.014363008551299572, 0.021802784875035286, 0.006530041806399822, 0.060474202036857605, 0.022080330178141594, 0.02292838878929615, -0.014054623432457447, 0.012435604818165302, -0.01944364234805107, -0.002921943087130785, 0.005539356730878353, 0.025195013731718063, 0.0434822142124176, -0.0385480634868145, 0.003719888161867857, 0.02960491180419922, -0.014902681112289429, 0.03595763444900513, -0.010654685087502003, -0.010222946293652058, 0.02982078120112419, 0.029389042407274246, -0.022820454090833664, -0.03466241806745529, 0.03601931035518646, 0.017439143732190132, 0.005994223989546299, -0.01791713945567608, 0.060104139149188995, -0.020723437890410423, 0.0049727000296115875, 0.05045170